<a href="https://colab.research.google.com/github/ian-shade/llm-adaptation-fintech/blob/main/demo_four_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Four systems, one question — live demo

**MSc AI Dissertation — Ihsan Abourshaid, Manchester Metropolitan University**
Companion to *Fine-Tuning vs RAG vs Hybrid: A Comparative Framework for Domain-Specific LLM Adaptation in Financial Services*.

This notebook is the **demonstration artefact**, not the experiment. It loads the frozen
`v9` run from Drive and puts one question to all four systems at once:

| | adapter | retrieval |
|---|---|---|
| **D — Baseline** | off | off |
| **A — RAG** | off | on |
| **B — Fine-tuned** | on | off |
| **C — Hybrid** | on | on |

All four are served from **one** 4-bit Qwen2.5-7B-Instruct. The QLoRA adapter is attached once and
switched off with `disable_adapter()` for D and A, so the two binary factors of the experimental
design are the only things that vary — exactly as in the pipeline notebook. The prompt builder,
decoding config, `TOP_K` and `MAX_NEW_TOKENS` are copied verbatim from `dissertation_full_pipeline_v9.1`.

### Before you run

1. **Runtime → Change runtime type → T4 GPU.** The 4-bit model needs ~6 GB free.
2. Your Drive must contain the finished `v9` run:
   ```
   MyDrive/dissertation/runs/v9/faiss_flatip.index
   MyDrive/dissertation/runs/v9/corpus_indexed.csv
   MyDrive/dissertation/runs/v9/qlora_adapter/
   MyDrive/dissertation/runs/v9/results_*.csv        (for the Results tab)
   MyDrive/dissertation/data/3banks_eval300.jsonl    (to load held-out questions)
   ```
3. **Runtime → Run all**, then wait for the launch cell. First run downloads the base model
   (~5.5 GB), so allow **8–10 minutes**. Every cell after that is seconds.

> **Demo day:** run this to completion *before* the call starts, ask one throwaway question to warm
> the GPU, and leave the tab open and active. Cold-starting in front of examiners costs you the
> whole demo slot.

In [ ]:
# ============================================================
# 0. CONFIGURATION — the only cell you should need to edit
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/dissertation"
RUN_NAME   = "v9"                      # the run whose index + adapter you are demonstrating

RUN_DIR    = f"{DRIVE_ROOT}/runs/{RUN_NAME}"
EVAL_PATH  = f"{DRIVE_ROOT}/data/3banks_eval300.jsonl"

# ---- models (identical to the pipeline) ----
BASE_MODEL  = "Qwen/Qwen2.5-7B-Instruct"
EMBED_MODEL = "BAAI/bge-base-en-v1.5"

# ---- retrieval + generation (identical to the pipeline) ----
TOP_K          = 6
MAX_NEW_TOKENS = 512
MAX_PROMPT_LEN = 3584

SYSTEM_PROMPT  = ("You are a financial analysis assistant answering questions about the "
                  "2023 annual reports of HSBC Holdings plc, Lloyds Banking Group plc and "
                  "NatWest Group plc. Answer concisely and factually, and make sure the "
                  "figure you give belongs to the bank named in the question. If you "
                  "cannot determine the answer from available information, say so explicitly.")

BANK_LABELS = {"hsbc": "HSBC", "lloyds": "Lloyds", "natwest": "NatWest"}

# ---- demo behaviour ----
SHARE_LINK  = True    # True = also print a public https://...gradio.live URL you can paste into
                      # the Teams chat so an examiner can try it on their own machine (72h life).
WARM_UP     = True    # run one throwaway generation per path at load, so the first real
                      # question is not the slow one.

print(f"Run directory: {RUN_DIR}")

Run directory: /content/drive/MyDrive/dissertation/runs/v9


## 1. Environment

In [ ]:
%%capture install_log
!pip install -q -U "gradio>=5" transformers accelerate bitsandbytes peft sentence-transformers faiss-cpu

In [ ]:
# Version report. Once the demo works end to end, freeze these exact versions —
# Colab updates its base image and a transformers/peft bump can break a notebook
# that worked last month. Copy the printed line into the install cell above as
# pinned ==versions, and re-test the day before the presentation.
import importlib
for _p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate",
           "sentence_transformers", "faiss", "gradio"]:
    try:
        _m = importlib.import_module(_p)
        print(f"{_p:24s} {getattr(_m, '__version__', 'n/a')}")
    except Exception as _e:
        print(f"{_p:24s} NOT IMPORTABLE — {_e}")

torch                    2.11.0+cu128
transformers             5.15.0
peft                     0.20.0
bitsandbytes             0.50.1
accelerate               1.14.0
sentence_transformers    5.7.0
faiss                    1.15.0
gradio                   6.24.0


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch

import re, gc, json, time, glob, string
import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.\n"
        "The 4-bit base model cannot be served from CPU at demo speed.")

_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name}  |  {_p.total_memory/1e9:.1f} GB total")

from google.colab import drive
drive.mount('/content/drive')

GPU: Tesla T4  |  15.6 GB total
Mounted at /content/drive


In [ ]:
# ---- preflight: fail here with a readable list, not three cells later ----
INDEX_PATH   = f"{RUN_DIR}/faiss_flatip.index"
CORPUS_PATH  = f"{RUN_DIR}/corpus_indexed.csv"
ADAPTER_DIR  = f"{RUN_DIR}/qlora_adapter"

REQUIRED = {
    "FAISS index":        INDEX_PATH,
    "indexed corpus":     CORPUS_PATH,
    "QLoRA adapter":      f"{ADAPTER_DIR}/adapter_config.json",
}
OPTIONAL = {
    "held-out eval set":  EVAL_PATH,
    "stored results":     f"{RUN_DIR}/results_baseline.csv",
    "figures":            f"{RUN_DIR}/figures",
}

missing = [name for name, path in REQUIRED.items() if not os.path.exists(path)]
for name, path in {**REQUIRED, **OPTIONAL}.items():
    mark = "found  " if os.path.exists(path) else "MISSING"
    print(f"  [{mark}] {name:20s} {path}")

if missing:
    raise FileNotFoundError(
        f"Cannot run the demo without: {', '.join(missing)}.\n"
        f"Check RUN_NAME (currently {RUN_NAME!r}) and that Drive finished mounting.")
print("\nPreflight OK.")

  [found  ] FAISS index          /content/drive/MyDrive/dissertation/runs/v9/faiss_flatip.index
  [found  ] indexed corpus       /content/drive/MyDrive/dissertation/runs/v9/corpus_indexed.csv
  [found  ] QLoRA adapter        /content/drive/MyDrive/dissertation/runs/v9/qlora_adapter/adapter_config.json
  [found  ] held-out eval set    /content/drive/MyDrive/dissertation/data/3banks_eval300.jsonl
  [found  ] stored results       /content/drive/MyDrive/dissertation/runs/v9/results_baseline.csv
  [found  ] figures              /content/drive/MyDrive/dissertation/runs/v9/figures

Preflight OK.


## 2. Retrieval

The index is **read from Drive, not rebuilt** — the demo must retrieve from the same vectors the
reported results came from. The embedder stays on the CPU, as in the pipeline: encoding one query
costs ~20 ms there, and leaving it on the GPU is what starves the 4-bit model load.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

corpus_df = pd.read_csv(CORPUS_PATH)
index     = faiss.read_index(INDEX_PATH)

assert index.ntotal == len(corpus_df), (
    f"Index has {index.ntotal} vectors but the corpus CSV has {len(corpus_df)} rows. "
    "These are from different runs — the demo would retrieve the wrong text.")

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(question, k=TOP_K):
    q = embedder.encode([BGE_QUERY_PREFIX + question],
                        normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q, k)
    hits = []
    for s, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        row = corpus_df.iloc[int(i)]
        hits.append({"chunk_id": str(row["chunk_id"]),
                     "bank": str(row.get("bank", "unknown")),
                     "score": float(s),
                     "text": str(row["text"])})
    return hits

print(f"{index.ntotal} vectors  |  units per bank:",
      {BANK_LABELS.get(b, b): int(n) for b, n in corpus_df["bank"].value_counts().items()})
_probe = retrieve("What was HSBC's total operating income in 2023?")
print(f"probe: top-1 {_probe[0]['bank']} @ {_probe[0]['score']:.3f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

113 vectors  |  units per bank: {'HSBC': 48, 'Lloyds': 33, 'NatWest': 32}
probe: top-1 hsbc @ 0.804


## 3. One model, four systems

`PeftModel` wraps the quantised base once. `disable_adapter()` gives the unadapted model back for
D and A, so there is a single set of weights in VRAM and no possibility of the two paths drifting
apart. The tokeniser saved beside the adapter is used for B and C — the chat template must be the
one the adapter was trained under.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

gc.collect(); torch.cuda.empty_cache()
_free = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_free:.2f} GB free before load")
if _free < 6.0:
    raise RuntimeError("Under 6 GB free. Runtime -> Restart session, then Run all.")

print("Loading base model (first run downloads ~5.5 GB) ...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=BNB_CONFIG,
    device_map={"": 0}, torch_dtype=torch.float16)
base.eval()

tok_base = AutoTokenizer.from_pretrained(BASE_MODEL)
try:
    tok_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    print("Tokeniser for B/C: loaded from the adapter directory (trained template).")
except Exception as e:
    tok_ft = tok_base
    print(f"Tokeniser for B/C: falling back to the base tokeniser ({e}).")

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print(f"Adapter attached. {torch.cuda.mem_get_info()[0]/1e9:.2f} GB free after load.")

15.53 GB free before load
Loading base model (first run downloads ~5.5 GB) ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokeniser for B/C: loaded from the adapter directory (trained template).
Adapter attached. 9.62 GB free after load.


In [ ]:
# ---- prompt construction: copied verbatim from the pipeline ----
LETTERS = list(string.ascii_uppercase)

TYPE_INSTRUCTIONS = {
    "true_false":      'Answer with exactly one word: "True" or "False".',
    "single_choice":   "Answer with the letter of the single correct option, and nothing else.",
    "multiple_choice": 'Answer with the letters of ALL correct options, separated by commas (e.g. "A, C").',
    "short_answer":    "Answer with the exact figure or short phrase only — no explanation.",
    "open_ended":      "",
}

def format_question(row):
    q = str(row["question"]).strip()
    if row.get("options_list"):
        opts = "\n".join(f"{LETTERS[i]}. {o}" for i, o in enumerate(row["options_list"]))
        q = q + "\n\nOptions:\n" + opts
    return (q + "\n\n" + TYPE_INSTRUCTIONS.get(row.get("question_type", "open_ended"), "")).strip()

def build_messages(row, context_hits=None):
    user_q = format_question(row)
    if context_hits:
        ctx = "\n\n".join(f"[{h['chunk_id']}] {h['text']}" for h in context_hits)
        user = ("Use ONLY the following extracts from the annual report to answer. "
                "If the answer is not contained in the extracts, say so explicitly."
                "\n\n--- EXTRACTS ---\n" + ctx + "\n--- END EXTRACTS ---\n\n" + user_q)
    else:
        user = user_q
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]

# ---- abstention detector: copied verbatim from the pipeline scorer ----
ABSTAIN_RE = re.compile(
    r"(not (contained|present|provided|available|specified|mentioned|found|stated|disclosed)|"
    r"cannot (be )?(determine|answer|find)|can't (determine|answer|find)|"
    r"do not have (enough|sufficient|the)|insufficient information|"
    r"no information|not in the extracts|does not (contain|include|mention|provide)|unable to|"
    r"not (disclosed|reported|broken out|separately)|is not given|"
    r"cannot be (established|derived|calculated)|not possible to determine|"
    r"there is no (figure|mention|breakdown)|"
    r"(don't|do not) know|not covered (in|by))", re.I)

CONDITIONS = [
    {"key": "baseline",  "letter": "D", "name": "Baseline",   "adapter": False, "retrieval": False},
    {"key": "rag",       "letter": "A", "name": "RAG",        "adapter": False, "retrieval": True },
    {"key": "finetuned", "letter": "B", "name": "Fine-tuned", "adapter": True,  "retrieval": False},
    {"key": "hybrid",    "letter": "C", "name": "Hybrid",     "adapter": True,  "retrieval": True },
]

@torch.no_grad()
def answer_one(cond, row):
    """One condition's answer to one question. Returns (text, meta, hits)."""
    t0   = time.time()
    hits = retrieve(row["question"]) if cond["retrieval"] else None
    tok  = tok_ft if cond["adapter"] else tok_base

    prompt = tok.apply_chat_template(build_messages(row, hits),
                                     tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt",
                 truncation=True, max_length=MAX_PROMPT_LEN).to(model.device)

    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                      temperature=None, top_p=None, top_k=None,
                      pad_token_id=tok.eos_token_id)
    if cond["adapter"]:
        out = model.generate(**inputs, **gen_kwargs)
    else:
        with model.disable_adapter():
            out = model.generate(**inputs, **gen_kwargs)

    new_ids = out[0][inputs["input_ids"].shape[1]:]
    text    = tok.decode(new_ids, skip_special_tokens=True).strip()

    q_bank = str(row.get("bank", "") or "")
    meta = {"latency": time.time() - t0,
            "tokens": int(new_ids.shape[0]),
            "capped": int(new_ids.shape[0]) >= MAX_NEW_TOKENS,
            "abstained": bool(ABSTAIN_RE.search(text))}
    if hits:
        meta["top1"] = max(h["score"] for h in hits)
        if q_bank in BANK_LABELS:
            meta["bank_prec"] = sum(h["bank"] == q_bank for h in hits) / len(hits)
    return text, meta, hits

print("Inference paths ready.")

Inference paths ready.


In [ ]:
if WARM_UP:
    _row = {"question": "What was NatWest's total income in 2023?",
            "question_type": "short_answer", "bank": "natwest"}
    for _c in (CONDITIONS[0], CONDITIONS[3]):          # one adapter-off, one adapter-on
        _t, _m, _ = answer_one(_c, _row)
        print(f"warm-up {_c['letter']}: {_m['latency']:.1f}s  ->  {_t[:70]!r}")
    print("\nGPU warm. First question in the demo will now run at normal speed.")

warm-up D: 2.1s  ->  '£14,895m'
warm-up C: 8.3s  ->  '**14,752**'

GPU warm. First question in the demo will now run at normal speed.


## 4. The stored run — for the Results and Numbers tabs

Read-only. These are the same `results_*.csv` files Chapter 6 was built from, so anything the
Results tab shows can be traced to a named file in the run directory.

In [ ]:
COND_KEYS   = [c["key"] for c in CONDITIONS]
COND_LETTER = {c["key"]: c["letter"] for c in CONDITIONS}
COND_TITLE  = {c["key"]: f'{c["letter"]} — {c["name"]}' for c in CONDITIONS}
OBJECTIVE_TYPES = ["true_false", "single_choice", "multiple_choice", "short_answer"]

# ---- held-out eval set (for "load a question the examiners can check") ----
def load_eval(path):
    """Minimal reader for the v8/v9 eval schema. qids are synthesised from row
    order exactly as the pipeline does, so they line up with results_*.csv."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows).rename(columns={"evalId": "qid", "questionType": "question_type"})
    if "qid" not in df.columns:
        df["qid"] = [f"E{i+1:04d}" for i in range(len(df))]
    tag_re = re.compile(r"([a-z_]+):([^|]+)", re.I)
    def bank_of(t):
        d = {k.strip().lower(): v.strip() for k, v in tag_re.findall(str(t or ""))}
        b = d.get("bank", "").lower()
        return b if b in BANK_LABELS else "unknown"
    df["bank"] = df.get("tags", pd.Series([""] * len(df))).map(bank_of)

    tf = {"✅": "true", "❌": "false"}
    def ref_of(r):
        a = str(r.get("correctAnswer", "")).strip()
        if r["question_type"] == "true_false":
            return tf.get(a, a.lower())
        if r["question_type"] == "multiple_choice":
            try:    return ",".join(sorted(json.loads(a)))
            except Exception: return a
        return a
    def opts_of(r):
        if r["question_type"] in ("single_choice", "multiple_choice"):
            try:    return json.loads(str(r.get("options", "")))
            except Exception: return None
        return None
    df["reference"]    = df.apply(ref_of, axis=1)
    df["options_list"] = df.apply(opts_of, axis=1)
    return df[["qid", "question_type", "bank", "question", "options_list", "reference"]]

try:
    eval_df = load_eval(EVAL_PATH)
    print(f"Held-out set: {len(eval_df)} questions")
except Exception as e:
    eval_df = pd.DataFrame(columns=["qid", "question_type", "bank", "question",
                                    "options_list", "reference"])
    print(f"Held-out set unavailable ({e}). Free-text questions still work.")

# ---- stored predictions, merged one row per question ----
def load_stored():
    frames = {}
    for k in COND_KEYS:
        p = f"{RUN_DIR}/results_{k}.csv"
        if os.path.exists(p):
            frames[k] = pd.read_csv(p)
    if not frames:
        return pd.DataFrame()

    first = frames[COND_KEYS[0] if COND_KEYS[0] in frames else list(frames)[0]]
    keep  = [c for c in ["qid", "question_type", "bank", "question", "reference"]
             if c in first.columns]
    out = first[keep].copy()
    out["qid"] = out["qid"].astype(str)

    for k, df in frames.items():
        df = df.copy()
        df["qid"] = df["qid"].astype(str)
        cols = {"prediction": f"pred_{k}"}
        if "correct" in df.columns:
            cols["correct"] = f"ok_{k}"
        if "retrieved_ids" in df.columns:
            cols["retrieved_ids"] = f"ids_{k}"
        if "latency_s" in df.columns:
            cols["latency_s"] = f"lat_{k}"
        out = out.merge(df[["qid"] + list(cols)].rename(columns=cols), on="qid", how="left")

    ok_cols = [f"ok_{k}" for k in COND_KEYS if f"ok_{k}" in out.columns]
    if ok_cols:
        out["n_correct"] = out[ok_cols].fillna(0).astype(float).round().sum(axis=1).astype(int)
        out["pattern"] = out.apply(
            lambda r: "".join(COND_LETTER[k] for k in COND_KEYS
                              if f"ok_{k}" in out.columns
                              and float(r.get(f"ok_{k}") or 0) >= 0.5) or "none",
            axis=1)
    return out

stored = load_stored()
if len(stored):
    obj = stored[stored["question_type"].isin(OBJECTIVE_TYPES)]
    ACC = {k: (obj[f"ok_{k}"].astype(float).mean() if f"ok_{k}" in obj.columns else None)
           for k in COND_KEYS}
    print(f"Stored run: {len(stored)} questions, {len(obj)} objective")
    print("Objective accuracy:", {COND_LETTER[k]: (f"{v*100:.1f}%" if v is not None else "n/a")
                                  for k, v in ACC.items()})
else:
    ACC = {k: None for k in COND_KEYS}
    print("No results_*.csv in the run directory — the Results tab will be empty.")

Held-out set: 300 questions
Stored run: 300 questions, 240 objective
Objective accuracy: {'D': '22.1%', 'A': '64.2%', 'B': '27.5%', 'C': '66.2%'}


## 5. Interface

In [ ]:
CSS = r"""
@import url('https://fonts.googleapis.com/css2?family=Instrument+Serif:ital@0;1&family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500;600&display=swap');

:root, .dark {
  --ground:   #0E161D;
  --surface:  #16212B;
  --surface-2:#1D2B36;
  --rule:     #2B3D4B;
  --rule-2:   #3A4E5E;
  --text:     #E8EFF5;
  --text-2:   #A6B9C8;
  --text-3:   #74899B;
  --d: #93A5B3;   /* D Baseline   */
  --a: #4FA3E3;   /* A RAG        */
  --b: #E85F68;   /* B Fine-tuned */
  --c: #3FC07A;   /* C Hybrid     */
}

/* ---------- shell: full bleed, dark ---------- */
.gradio-container {
  background: var(--ground) !important;
  max-width: none !important;
  width: 100% !important;
  padding: 0 26px 40px 26px !important;
  color: var(--text) !important;
}
body, gradio-app, .gradio-container .main, .gradio-container .wrap {
  background: var(--ground) !important;
}
.gradio-container, .gradio-container * {
  font-family: 'IBM Plex Sans', ui-sans-serif, system-ui, sans-serif;
}
footer { display: none !important; }

/* ---------- Gradio's own chrome, restyled ---------- */
.gradio-container .block,
.gradio-container .form,
.gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container button { border-radius: 0 !important; }

.gradio-container .block,
.gradio-container .form {
  background: var(--surface) !important;
  border-color: var(--rule) !important;
  box-shadow: none !important;
}
.gradio-container label,
.gradio-container label span,
.gradio-container .block-title,
.gradio-container span[data-testid="block-info"] { color: var(--text-2) !important; }

.gradio-container input,
.gradio-container textarea,
.gradio-container .wrap-inner,
.gradio-container .secondary-wrap {
  background: var(--surface-2) !important;
  color: var(--text) !important;
  border-color: var(--rule) !important;
}
.gradio-container input::placeholder,
.gradio-container textarea::placeholder { color: var(--text-3) !important; }

/* dropdown menus */
.gradio-container ul.options,
.gradio-container .options { background: var(--surface-2) !important; border-color: var(--rule) !important; }
.gradio-container ul.options li,
.gradio-container .options .item { color: var(--text) !important; }
.gradio-container ul.options li:hover,
.gradio-container .options .item:hover { background: var(--rule) !important; }

.gradio-container button.primary {
  background: var(--a) !important; color: #06131E !important;
  border: 1px solid var(--a) !important;
  font-family: 'IBM Plex Mono', ui-monospace, monospace !important;
  text-transform: uppercase; letter-spacing: 0.07em; font-size: 12px !important;
  font-weight: 600 !important;
}
.gradio-container button.primary:hover { filter: brightness(1.12); }

.gradio-container .tab-nav { border-color: var(--rule) !important; }
.gradio-container .tab-nav button {
  font-family: 'IBM Plex Mono', ui-monospace, monospace !important;
  font-size: 12px !important; letter-spacing: 0.06em; text-transform: uppercase;
  color: var(--text-3) !important; background: transparent !important;
  border-color: transparent !important;
}
.gradio-container .tab-nav button.selected {
  color: var(--text) !important;
  border-bottom: 2px solid var(--a) !important;
}

/* Gradio's dataframe */
.gradio-container table { background: var(--surface) !important; }
.gradio-container table thead th,
.gradio-container .table-wrap thead th {
  background: var(--surface-2) !important; color: var(--text-2) !important;
  border-color: var(--rule) !important;
  font-family: 'IBM Plex Mono', ui-monospace, monospace !important;
  font-size: 11px !important; letter-spacing: 0.05em; text-transform: uppercase;
}
.gradio-container table tbody td,
.gradio-container .table-wrap tbody td {
  background: var(--surface) !important; color: var(--text) !important;
  border-color: var(--rule) !important; font-size: 13px !important;
}
.gradio-container table tbody tr:hover td { background: var(--surface-2) !important; }
.gradio-container .table-wrap { border-color: var(--rule) !important; }

/* my own HTML blocks provide their own surfaces — strip Gradio's card off them */
.gradio-container .block:has(> .html-container) {
  background: transparent !important; border: none !important;
  padding: 0 !important; box-shadow: none !important;
}
.gradio-container .html-container { background: transparent !important; padding: 0 !important; }
#refslot:empty, #refslot { background: transparent !important; border: none !important; }

/* Gradio subdues loose spans inside HTML blocks; my components set their own colour */
.lane__foot span, .lane__foot b,
.chunk__head, .chunk__head span,
.refstrip span, .tile__lbl, .tile__sub,
.detail .qmeta, .detail .qmeta b,
.answer-row__txt, .verdict, .note { opacity: 1 !important; }
.lane__foot span { color: var(--text-2) !important; }
.lane__foot b    { color: var(--text)   !important; }
.lane__foot .warn { color: var(--b) !important; }
.lane__foot .abst { color: var(--a) !important; }
.chunk__head     { color: var(--text-3) !important; }
.chunk__head .bank { color: var(--text) !important; }
.refstrip .lbl   { color: var(--text-3) !important; }
.refstrip .val   { color: var(--text)   !important; }
.tile__lbl       { color: var(--text-2) !important; }
.tile__sub       { color: var(--text-3) !important; }
.tile__val       { color: var(--text)   !important; }
.detail .qmeta   { color: var(--text-3) !important; }
.answer-row__txt { color: var(--text)   !important; }
.note            { color: var(--text-2) !important; }
.datatable td    { color: var(--text)   !important; }
.datatable th    { color: var(--text-2) !important; }

/* accordions */
.gradio-container .label-wrap,
.gradio-container .label-wrap span { color: var(--text) !important; }

/* ---------- masthead ---------- */
.masthead {
  border-bottom: 2px solid var(--rule-2);
  padding: 10px 0 12px 0;
  margin-bottom: 2px;
}
.masthead h1 {
  font-family: 'Instrument Serif', Georgia, serif;
  font-size: 34px; line-height: 1.05; font-weight: 400;
  color: var(--text); margin: 0 0 5px 0; letter-spacing: -0.01em;
}
.masthead h1 em { font-style: italic; color: var(--a); }
.masthead p { font-size: 13px; color: var(--text-2); margin: 0; max-width: 92ch; }
.masthead .rule-note {
  font-family: 'IBM Plex Mono', ui-monospace, monospace;
  font-size: 11px; letter-spacing: 0.08em; text-transform: uppercase;
  color: var(--text-3); margin-top: 8px;
}

/* ---------- lanes: the signature element ---------- */
.lanes { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }
@media (max-width: 1100px) { .lanes { grid-template-columns: repeat(2, 1fr); } }
@media (max-width: 620px)  { .lanes { grid-template-columns: 1fr; } }

.lane {
  background: var(--surface);
  border: 1px solid var(--rule);
  border-top: 4px solid var(--accent, var(--text-3));
  display: flex; flex-direction: column; min-height: 210px;
}
.lane[data-state="idle"]    { opacity: 0.5; }
.lane[data-state="working"] { animation: lanePulse 1.4s ease-in-out infinite; }
@keyframes lanePulse { 0%,100% { opacity: 1; } 50% { opacity: 0.6; } }
@media (prefers-reduced-motion: reduce) { .lane[data-state="working"] { animation: none; } }

.lane__head { padding: 12px 14px 10px 14px; border-bottom: 1px solid var(--rule); }
.lane__letter {
  font-family: 'Instrument Serif', Georgia, serif;
  font-size: 34px; line-height: 1; color: var(--accent); margin-right: 8px;
}
.lane__name { font-size: 15px; font-weight: 600; color: var(--text); vertical-align: 6px; }
.lane__flags { display: flex; gap: 6px; margin-top: 10px; }
.flag {
  font-family: 'IBM Plex Mono', ui-monospace, monospace;
  font-size: 10px; letter-spacing: 0.06em; text-transform: uppercase;
  padding: 3px 6px; border: 1px solid var(--rule-2); font-weight: 500;
}
.flag--on  { background: var(--accent); color: #06131E; border-color: var(--accent); }
.flag--off { background: transparent; color: var(--text-3); }

.lane__body {
  padding: 14px; flex: 1; font-size: 14.5px; line-height: 1.55; color: var(--text);
  white-space: pre-wrap; word-break: break-word;
  max-height: 300px; overflow-y: auto;
}
.lane__body.is-empty { color: var(--text-3); font-style: italic; }

.lane__foot {
  border-top: 1px solid var(--rule); padding: 8px 14px;
  display: flex; flex-wrap: wrap; gap: 4px 14px;
  font-family: 'IBM Plex Mono', ui-monospace, monospace;
  font-size: 11px; color: var(--text-2);
}
.lane__foot b { color: var(--text); font-weight: 600; }
.lane__foot .warn { color: var(--b); }
.lane__foot .abst { color: var(--a); }

/* ---------- reference strip ---------- */
.refstrip {
  background: var(--surface-2); border-left: 4px solid var(--c);
  color: var(--text); padding: 10px 14px; margin: 6px 0;
  display: flex; gap: 14px; align-items: baseline; flex-wrap: wrap;
}
.refstrip .lbl {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 10px;
  letter-spacing: 0.1em; text-transform: uppercase; color: var(--text-3);
}
.refstrip .val {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 14px; color: var(--text);
}

/* ---------- retrieved extracts ---------- */
.chunk {
  border-left: 3px solid var(--a); background: var(--surface-2);
  padding: 8px 12px; margin-bottom: 10px;
}
.chunk__head {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 11px;
  color: var(--text-3); margin-bottom: 5px;
}
.chunk__head .bank { font-weight: 600; color: var(--text); }
.chunk__text {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 11.5px;
  line-height: 1.55; color: var(--text-2); white-space: pre-wrap;
  max-height: 170px; overflow-y: auto;
}

/* ---------- metric tiles ---------- */
.tiles { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 6px; }
@media (max-width: 900px) { .tiles { grid-template-columns: repeat(2, 1fr); } }
.tile {
  background: var(--surface); border: 1px solid var(--rule);
  border-left: 4px solid var(--accent); padding: 16px;
}
.tile__lbl { font-size: 12px; color: var(--text-2); margin-bottom: 8px; }
.tile__val {
  font-family: 'IBM Plex Mono', ui-monospace, monospace;
  font-size: 34px; font-weight: 600; color: var(--text); line-height: 1;
}
.tile__sub { font-size: 11px; color: var(--text-3); margin-top: 8px; }

.datatable { width: 100%; border-collapse: collapse; background: var(--surface); }
.datatable th, .datatable td {
  border: 1px solid var(--rule); padding: 9px 12px; font-size: 13.5px;
  text-align: left; color: var(--text);
}
.datatable th {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 10.5px;
  letter-spacing: 0.07em; text-transform: uppercase; color: var(--text-2);
  background: var(--surface-2); font-weight: 500;
}
.datatable td.n { font-family: 'IBM Plex Mono', ui-monospace, monospace; }
.datatable td.best { font-weight: 600; color: #FFFFFF; }

/* ---------- detail panel ---------- */
.detail { background: var(--surface); border: 1px solid var(--rule); padding: 18px; }
.detail h3 {
  font-family: 'Instrument Serif', Georgia, serif; font-weight: 400;
  font-size: 23px; margin: 0 0 6px 0; color: var(--text);
}
.detail .qmeta {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 11px;
  color: var(--text-3); margin-bottom: 14px;
  letter-spacing: 0.05em; text-transform: uppercase;
}
.detail .qmeta b { color: var(--c); }
.answer-row { display: flex; gap: 14px; padding: 11px 0; border-top: 1px solid var(--rule); }
.answer-row__tag {
  flex: 0 0 140px; font-family: 'IBM Plex Mono', ui-monospace, monospace;
  font-size: 12px; font-weight: 600;
}
.answer-row__txt {
  flex: 1; font-size: 13.5px; line-height: 1.55; white-space: pre-wrap; color: var(--text);
}
.verdict { font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 11px; flex: 0 0 70px; }
.verdict--y { color: var(--c); font-weight: 600; }
.verdict--n { color: var(--b); }

/* ---------- figures ---------- */
.figwrap { margin: 0 0 26px 0; }
.figwrap__cap {
  font-family: 'IBM Plex Mono', ui-monospace, monospace; font-size: 11px;
  letter-spacing: 0.06em; text-transform: uppercase; color: var(--text-3);
  margin-bottom: 6px;
}
.figwrap img {
  width: 100%; height: auto; display: block;
  background: #FFFFFF; border: 1px solid var(--rule); padding: 10px;
}

.note {
  font-size: 12.5px; color: var(--text-2); border-left: 3px solid var(--rule-2);
  padding-left: 12px; margin: 10px 0;
}
"""

print(f'{len(CSS)} characters of stylesheet')

12374 characters of stylesheet


In [ ]:
import html as _html

COND_ACCENT = {"baseline": "var(--d)", "rag": "var(--a)",
               "finetuned": "var(--b)", "hybrid": "var(--c)"}
COND_HEX    = {"baseline": "#93A5B3", "rag": "#4FA3E3",
               "finetuned": "#E85F68", "hybrid": "#3FC07A"}

def _esc(t):
    return _html.escape(str(t if t is not None else ""))

def lane_html(cond, state="idle", answer=None, meta=None):
    """One condition lane. state: idle | working | done."""
    accent = COND_ACCENT[cond["key"]]
    ad = "on" if cond["adapter"] else "off"
    rt = "on" if cond["retrieval"] else "off"
    flags = (f'<span class="flag flag--{ad}">adapter {ad}</span>'
             f'<span class="flag flag--{rt}">retrieval {rt}</span>')

    if state == "idle":
        body = '<div class="lane__body is-empty">waiting</div>'
    elif state == "working":
        body = '<div class="lane__body is-empty">reading&hellip;</div>'
    else:
        body = f'<div class="lane__body">{_esc(answer).strip() or "(empty answer)"}</div>'

    foot = ""
    if meta:
        bits = []
        if meta.get("latency") is not None:
            bits.append(f'<span><b>{meta["latency"]:.1f}s</b></span>')
        if meta.get("tokens") is not None:
            bits.append(f'<span>{meta["tokens"]} tok</span>')
        if meta.get("capped"):
            bits.append('<span class="warn">hit token cap</span>')
        if meta.get("abstained"):
            bits.append('<span class="abst">abstained</span>')
        if meta.get("bank_prec") is not None:
            bits.append(f'<span>bank precision {meta["bank_prec"]*100:.0f}%</span>')
        if meta.get("top1") is not None:
            bits.append(f'<span>top-1 {meta["top1"]:.2f}</span>')
        foot = '<div class="lane__foot">' + "".join(bits) + "</div>"

    return (f'<div class="lane" data-state="{state}" style="--accent:{accent}">'
            f'<div class="lane__head">'
            f'<span class="lane__letter">{cond["letter"]}</span>'
            f'<span class="lane__name">{cond["name"]}</span>'
            f'<div class="lane__flags">{flags}</div>'
            f'</div>{body}{foot}</div>')


def chunks_html(hits):
    if not hits:
        return ('<div class="note">Retrieval only runs for A and C. '
                'Ask a question to see what the index returned.</div>')
    parts = []
    for i, h in enumerate(hits, 1):
        parts.append(
            f'<div class="chunk">'
            f'<div class="chunk__head">{i}. <span class="bank">'
            f'{_esc(BANK_LABELS.get(h["bank"], h["bank"]))}</span> &middot; '
            f'{_esc(h["chunk_id"])} &middot; similarity {h["score"]:.3f}</div>'
            f'<div class="chunk__text">{_esc(h["text"])}</div></div>')
    return "".join(parts)


def reference_html(row):
    if not row or not row.get("reference"):
        return ""
    return ('<div class="refstrip">'
            f'<span class="lbl">held-out question</span>'
            f'<span class="val">{_esc(row.get("qid",""))}</span>'
            f'<span class="lbl">gold answer</span>'
            f'<span class="val">{_esc(row["reference"])}</span>'
            '</div>')

print('Renderers ready.')

Renderers ready.


In [ ]:
import gradio as gr

# ------------------------------------------------------------------ ask
def ask(question, qtype, options_text, ref_state):
    """Generator: yields after each system finishes, so the lanes fill in
    sequentially and the latency difference between D and C is visible live."""
    question = (question or "").strip()
    lanes = [lane_html(c, "idle") for c in CONDITIONS]

    if not question:
        yield (*lanes, chunks_html(None),
               '<div class="note">Type a question first.</div>')
        return

    opts = [o.strip() for o in (options_text or "").split("\n") if o.strip()]
    row = {"question": question, "question_type": qtype,
           "options_list": opts or None,
           "bank": (ref_state or {}).get("bank", "")}

    ref_html = reference_html(ref_state if (ref_state or {}).get("question", "") == question else None)
    shown_hits = None

    for i, cond in enumerate(CONDITIONS):
        lanes[i] = lane_html(cond, "working")
        yield (*lanes, chunks_html(shown_hits), ref_html)

        try:
            text, meta, hits = answer_one(cond, row)
        except Exception as e:
            lanes[i] = lane_html(cond, "done", f"[generation failed: {e}]", None)
            yield (*lanes, chunks_html(shown_hits), ref_html)
            continue

        if hits and shown_hits is None:
            shown_hits = hits
        lanes[i] = lane_html(cond, "done", text, meta)
        yield (*lanes, chunks_html(shown_hits), ref_html)


# ------------------------------------------------------------------ load a held-out question
EVAL_CHOICES = [
    f'{r.qid} · {BANK_LABELS.get(r.bank, r.bank)} · {r.question_type} · {str(r.question)[:70]}'
    for r in eval_df.itertuples()
] if len(eval_df) else []

def load_question(choice):
    if not choice:
        return gr.update(), gr.update(), gr.update(), None, ""
    qid = choice.split(" · ")[0]
    m = eval_df[eval_df["qid"] == qid]
    if not len(m):
        return gr.update(), gr.update(), gr.update(), None, ""
    r = m.iloc[0].to_dict()
    opts = "\n".join(r["options_list"]) if r.get("options_list") else ""
    return (gr.update(value=r["question"]), gr.update(value=r["question_type"]),
            gr.update(value=opts), r, reference_html(r))


# ------------------------------------------------------------------ results explorer
PATTERN_FILTERS = {
    "everything": None,
    "retrieval only (A + C, neither retrieval-free system)": lambda p: set(p) == {"A", "C"},
    "no system got it right": lambda p: p == "none",
    "every system got it right": lambda p: set(p) == {"D", "A", "B", "C"},
    "the adapter changed the outcome (B or C only)": lambda p: set(p) & {"B", "C"} and not set(p) & {"D", "A"},
}

def filter_rows(bank, qtype, pattern_label):
    if not len(stored):
        return pd.DataFrame({"note": ["No stored results in this run directory."]})
    df = stored
    if bank != "all banks":
        df = df[df["bank"] == bank]
    if qtype != "all types":
        df = df[df["question_type"] == qtype]
    fn = PATTERN_FILTERS.get(pattern_label)
    if fn is not None and "pattern" in df.columns:
        df = df[df["pattern"].map(fn)]
    view = pd.DataFrame({
        "qid": df["qid"],
        "bank": df["bank"].map(lambda b: BANK_LABELS.get(b, b)),
        "type": df["question_type"],
        "correct (D A B C)": df.get("pattern", pd.Series(["—"] * len(df), index=df.index)),
        "question": df["question"].astype(str).str.slice(0, 110),
    })
    return view

def show_detail(view, evt: gr.SelectData):
    """The displayed table is passed in, so the row clicked is always the row
    shown — resolving it by re-running the filters would desynchronise the
    moment a filter changed between render and click."""
    if view is None or "qid" not in getattr(view, "columns", []):
        return '<div class="note">Nothing selected.</div>'
    row_ix = evt.index[0] if isinstance(evt.index, (list, tuple)) else evt.index
    if row_ix is None or row_ix >= len(view):
        return '<div class="note">Nothing selected.</div>'
    qid = str(view.iloc[row_ix]["qid"])
    match = stored[stored["qid"].astype(str) == qid]
    if not len(match):
        return '<div class="note">Nothing selected.</div>'
    r = match.iloc[0]

    rows = []
    for c in CONDITIONS:
        k = c["key"]
        ok = r.get(f"ok_{k}")
        verdict = ("—" if pd.isna(ok) else
                   ('<span class="verdict verdict--y">correct</span>' if float(ok) >= 0.5
                    else '<span class="verdict verdict--n">wrong</span>'))
        rows.append(
            f'<div class="answer-row">'
            f'<div class="answer-row__tag" style="color:{COND_HEX[k]}">'
            f'{_esc(COND_TITLE[k])}</div>'
            f'<div class="answer-row__txt">{_esc(r.get(f"pred_{k}", ""))}</div>'
            f'{verdict}</div>')

    return ('<div class="detail">'
            f'<h3>{_esc(r["question"])}</h3>'
            f'<div class="qmeta">{_esc(r["qid"])} &middot; '
            f'{_esc(BANK_LABELS.get(r["bank"], r["bank"]))} &middot; '
            f'{_esc(r["question_type"])} &middot; gold: '
            f'<b>{_esc(r.get("reference", ""))}</b></div>'
            + "".join(rows) +
            '<div class="note">Stored predictions from the v9 run. Correctness is the '
            'harness verdict recorded in results_*.csv, not a live re-score.</div></div>')


# ------------------------------------------------------------------ numbers
def tiles_html():
    out = []
    for c in CONDITIONS:
        v = ACC.get(c["key"])
        val = f"{v*100:.1f}%" if v is not None else "n/a"
        out.append(f'<div class="tile" style="--accent:{COND_ACCENT[c["key"]]}">'
                   f'<div class="tile__lbl">{c["letter"]} — {c["name"]}</div>'
                   f'<div class="tile__val">{val}</div>'
                   f'<div class="tile__sub">objective accuracy, n={len(stored[stored["question_type"].isin(OBJECTIVE_TYPES)]) if len(stored) else 0}</div>'
                   f'</div>')
    return '<div class="tiles">' + "".join(out) + "</div>"

HEADLINE_TABLE = """
<table class="datatable">
<tr><th>metric</th><th>D Baseline</th><th>A RAG</th><th>B Fine-tuned</th><th>C Hybrid</th></tr>
<tr><td>Objective accuracy (n=240)</td><td class="n">22.1%</td><td class="n">64.2%</td><td class="n">27.5%</td><td class="n best">66.3%</td></tr>
<tr><td>Open-ended, judge 1–5</td><td class="n">1.68</td><td class="n">2.98</td><td class="n">1.62</td><td class="n best">3.10</td></tr>
<tr><td>Open-ended, BERTScore</td><td class="n">0.837</td><td class="n">0.860</td><td class="n best">0.873</td><td class="n">0.872</td></tr>
<tr><td>Hallucination proxy</td><td class="n">76.3%</td><td class="n best">32.9%</td><td class="n">72.5%</td><td class="n">33.8%</td></tr>
<tr><td>Claim-level hallucination</td><td class="n">51.9%</td><td class="n best">36.7%</td><td class="n">85.4%</td><td class="n">47.3%</td></tr>
<tr><td>Abstention on unanswerable</td><td class="n">30.0%</td><td class="n best">76.7%</td><td class="n">0.0%</td><td class="n">0.0%</td></tr>
<tr><td>Median latency</td><td class="n">0.59 s</td><td class="n">5.69 s</td><td class="n">1.12 s</td><td class="n">6.69 s</td></tr>
<tr><td>GPU cost (Colab units)</td><td class="n">0.43</td><td class="n">1.52</td><td class="n">1.03</td><td class="n">2.46</td></tr>
</table>
<div class="note">v9.1 re-score. Retrieval bank-precision 100%, answer-in-context ceiling 86%,
extraction efficiency 75.5% (RAG) and 78.6% (Hybrid) — retrieval is not the bottleneck, reading is.</div>
"""

FIG_FILES = sorted(glob.glob(f"{RUN_DIR}/figures/*.png"))


# ------------------------------------------------------------------ layout
theme = gr.themes.Base(
    primary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("IBM Plex Sans"), "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "monospace"],
).set(
    # Set the LIGHT tokens to dark values too. The app is dark by design, and
    # this way it stays dark even if the browser or Colab is in light mode and
    # the force-dark script below is blocked.
    body_background_fill="#0E161D",          body_background_fill_dark="#0E161D",
    body_text_color="#E8EFF5",               body_text_color_dark="#E8EFF5",
    body_text_color_subdued="#A6B9C8",       body_text_color_subdued_dark="#A6B9C8",
    background_fill_primary="#16212B",       background_fill_primary_dark="#16212B",
    background_fill_secondary="#1D2B36",     background_fill_secondary_dark="#1D2B36",
    block_background_fill="#16212B",         block_background_fill_dark="#16212B",
    block_label_text_color="#A6B9C8",        block_label_text_color_dark="#A6B9C8",
    block_title_text_color="#A6B9C8",        block_title_text_color_dark="#A6B9C8",
    border_color_primary="#2B3D4B",          border_color_primary_dark="#2B3D4B",
    input_background_fill="#1D2B36",         input_background_fill_dark="#1D2B36",
    table_even_background_fill="#16212B",    table_even_background_fill_dark="#16212B",
    table_odd_background_fill="#1A2732",     table_odd_background_fill_dark="#1A2732",
)

# The app is designed dark. Without this it inherits whatever the browser or
# Colab is set to, and a light shell under a dark stylesheet is unreadable.
FORCE_DARK_JS = """
() => {
  const url = new URL(window.location);
  if (url.searchParams.get('__theme') !== 'dark') {
    url.searchParams.set('__theme', 'dark');
    window.location.replace(url.href);
  }
}
"""

# Gradio 6 moved `css` and `theme` from the Blocks constructor to launch().
# Branch on the version so this notebook runs on either.
_GR_MAJOR = int(gr.__version__.split(".")[0])
_blocks_kw, LAUNCH_KW = {"title": "Four systems, one question"}, {}
if _GR_MAJOR >= 6:
    LAUNCH_KW = {"theme": theme, "css": CSS, "js": FORCE_DARK_JS}
else:
    _blocks_kw.update(css=CSS, theme=theme, js=FORCE_DARK_JS)

with gr.Blocks(**_blocks_kw) as demo:
    gr.HTML(
        '<div class="masthead">'
        '<h1>Four systems, <em>one question</em></h1>'
        '<p>The same 4-bit Qwen2.5-7B-Instruct, differing only in whether the QLoRA adapter is '
        'attached and whether the retrieval index is consulted. Ask about the 2023 annual reports '
        'of HSBC, Lloyds or NatWest and watch all four answer.</p>'
        f'<div class="rule-note">run {RUN_NAME} &nbsp;·&nbsp; top-k {TOP_K} &nbsp;·&nbsp; '
        f'greedy decoding &nbsp;·&nbsp; max {MAX_NEW_TOKENS} new tokens</div></div>')

    with gr.Tabs():
        # ---------------------------------------------------------- ASK
        with gr.Tab("Ask"):
            ref_state = gr.State(None)

            with gr.Row():
                q_box = gr.Textbox(
                    label="Question", scale=5, lines=2,
                    placeholder="e.g. What was Lloyds Banking Group's net interest income in 2023?")
                q_type = gr.Dropdown(
                    label="Answer format", scale=1,
                    choices=["short_answer", "true_false", "single_choice",
                             "multiple_choice", "open_ended"],
                    value="short_answer")

            with gr.Row():
                ask_btn = gr.Button("Ask all four systems", variant="primary", scale=2)
                if EVAL_CHOICES:
                    eval_pick = gr.Dropdown(label="…or load one from the held-out set",
                                            choices=EVAL_CHOICES, value=None, scale=5)
                else:
                    eval_pick = gr.Dropdown(label="held-out set unavailable",
                                            choices=[], visible=False, scale=5)

            with gr.Accordion("Options (one per line — choice questions only)", open=False):
                opt_box = gr.Textbox(label="", lines=4, show_label=False)

            ref_html = gr.HTML("", elem_id="refslot")

            with gr.Row(elem_classes="lanes"):
                lane_out = [gr.HTML(lane_html(c, "idle")) for c in CONDITIONS]

            with gr.Accordion(f"What retrieval put in front of A and C (top {TOP_K})", open=False):
                ctx_html = gr.HTML(chunks_html(None))

            gr.HTML('<div class="note">Lanes fill in the order the systems finish, so the '
                    'latency difference is the demo, not a caption. Load a question from the '
                    'held-out set to see the gold answer alongside. Live answers are not '
                    'scored — correctness verdicts live in the next tab, where they come from '
                    'the harness that produced Chapter 6.</div>')

            ask_btn.click(ask, [q_box, q_type, opt_box, ref_state],
                          [*lane_out, ctx_html, ref_html])
            q_box.submit(ask, [q_box, q_type, opt_box, ref_state],
                         [*lane_out, ctx_html, ref_html])
            eval_pick.change(load_question, eval_pick,
                             [q_box, q_type, opt_box, ref_state, ref_html])

        # ---------------------------------------------------------- RESULTS
        with gr.Tab("All 300 questions"):
            gr.HTML('<div class="note">Every held-out question with all four stored answers. '
                    'The “retrieval only” filter is the 91-question block — the largest single '
                    'outcome pattern in the study, and the retrieval effect made visible at '
                    'question level.</div>')
            with gr.Row():
                f_bank = gr.Dropdown(label="Bank", value="all banks",
                                     choices=["all banks"] + list(BANK_LABELS))
                f_type = gr.Dropdown(label="Question type", value="all types",
                                     choices=["all types"] + OBJECTIVE_TYPES + ["open_ended"])
                f_pat = gr.Dropdown(label="Which systems got it right",
                                    value="everything", choices=list(PATTERN_FILTERS))
            table = gr.Dataframe(value=filter_rows("all banks", "all types", "everything"),
                                 interactive=False, wrap=True, max_height=520)
            detail = gr.HTML('<div class="note">Select a row to see all four answers.</div>')

            for ctrl in (f_bank, f_type, f_pat):
                ctrl.change(filter_rows, [f_bank, f_type, f_pat], table)
            table.select(show_detail, table, detail)

        # ---------------------------------------------------------- NUMBERS
        with gr.Tab("The numbers"):
            gr.HTML(tiles_html())
            gr.HTML(HEADLINE_TABLE)
        # ---------------------------------------------------------- FIGURES
        if FIG_FILES:
            with gr.Tab(f"Figures ({len(FIG_FILES)})"):
                gr.HTML('<div class="note">Every figure from the '
                        f'{RUN_NAME} run, full width, in file order — scroll the page. '
                        'Right-click to open one at full resolution.</div>')
                for _p in FIG_FILES:
                    gr.HTML(f'<div class="figwrap__cap">'
                            f'{_esc(os.path.basename(_p))}</div>')
                    gr.Image(value=_p, show_label=False, container=False, height=620)

print("App defined.")

App defined.


In [ ]:
# ============================================================
# LAUNCH
# ============================================================
# Do NOT present from the frame inside this cell. Colab gives cell output a
# fixed, scrolling box, so the lanes get cropped and you spend the demo
# scrolling. Open the app in its own browser tab instead — full width, no
# notebook chrome, and a browser tab is a cleaner thing to share on Teams
# than a Colab window.
#
#   * SHARE_LINK = True  -> a public https://....gradio.live URL is printed
#                           below. Open it in a new tab. You can also paste it
#                           into the Teams chat so an examiner can try the
#                           system on their own machine. Lives ~72 hours.
#   * SHARE_LINK = False -> run the pop-out cell underneath instead. It opens
#                           the same app in a new tab through Colab's own proxy,
#                           with no public tunnel.

INLINE_HEIGHT = 1200      # only affects the frame in this cell

demo.queue()              # one question at a time — four generations share one GPU

app, local_url, share_url = demo.launch(
    share=SHARE_LINK, height=INLINE_HEIGHT, debug=False, quiet=False, **LAUNCH_KW)

print("\n" + "=" * 72)
print("OPEN THIS IN A NEW BROWSER TAB AND PRESENT FROM THERE:")
print("  ", share_url or "(no share link — run the pop-out cell below)")
print("=" * 72)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0c30ff610d2a8a7b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



OPEN THIS IN A NEW BROWSER TAB AND PRESENT FROM THERE:
   https://e0c30ff610d2a8a7b5.gradio.live


In [ ]:
# Pop the app out into its own browser tab WITHOUT the public tunnel.
# Use this if share=False, or if gradio.live is blocked on your network.
from google.colab import output
output.serve_kernel_port_as_window(demo.server_port)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

---

## Demo-day checklist

**30 minutes before the call**

- [ ] Runtime → Change runtime type → **T4 GPU**
- [ ] Runtime → **Run all**, wait for the `gradio.live` URL the launch cell prints
- [ ] **Open that URL in its own browser tab** — never present from the frame inside the cell,
      it is a small scrolling box. Full-screen the tab (`⌃⌘F` on macOS) and zoom out one notch
      if the four lanes and the extracts panel don't quite fit
- [ ] In Teams, share **that browser tab**, not the whole screen and not Colab
- [ ] Ask one throwaway question so nothing is cold
- [ ] Leave the tab open and **active** — Colab disconnects an idle session after ~90 minutes
- [ ] Check your remaining Colab units

**Questions worth having ready**

| purpose | ask |
|---|---|
| the retrieval effect | a short-answer question one of the reports states plainly — A and C get it, D and B do not |
| the hard core | one of the 56 questions no system answers, so the ceiling is honest and yours to name first |
| the safety trade | an adversarial question about a figure that is not in any report — watch A abstain and C answer anyway |
| metric divergence | an open-ended question — B reads fluently and is wrong, which is the BERTScore-vs-judge story |
| the extraction gap | a question where the figure is visibly in the retrieved extracts and the answer is still wrong |

Rehearse narrating a **failure**. "The figure is on screen and it still missed it — that is the
extraction gap, and Section 6.7 decomposes it" is a stronger answer than a demo where nothing
goes wrong.

**If the GPU is unavailable on the day**, play the recording. Record it as soon as this notebook
runs cleanly, not the night before.